# Cooked or Not: grouped validation and submission

Run All with GPU and Internet enabled. Attach the competition data in Input. The notebook reads all 30 frames, extracts pretrained ResNet50 features once, trains five group-disjoint folds, and writes submission.csv in the Output tab.

The printed out-of-fold ROC-AUC is a validation estimate. Only a Kaggle submission can establish the public score; 0.90 is a target, not a guaranteed result.

In [ ]:
from pathlib import Path
import copy, gc, time
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights, resnet50

def find_data():
    candidates = [Path("/kaggle/input/competitions/cooked-or-not/crash_competition_data"),
                  Path.cwd() / "crash_competition_data",
                  Path.cwd().parent / "crash_competition_data"]
    candidates += list(Path("/kaggle/input").rglob("train_labels.csv")) if Path("/kaggle/input").exists() else []
    for p in candidates:
        root = p.parent if p.name == "train_labels.csv" else p
        if all((root / x).exists() for x in ("train_labels.csv", "sample_submission.csv", "train", "test")):
            return root
    raise FileNotFoundError("Attach the Cooked or Not competition data in Kaggle Input.")

DATA = find_data()
labels = pd.read_csv(DATA / "train_labels.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
assert list(labels.columns) == ["clip_id", "label", "group_id"]
assert list(sample.columns) == ["clip_id", "label"]
assert labels.clip_id.is_unique and sample.clip_id.is_unique
assert labels.groupby("group_id").label.nunique().eq(2).all()
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
print("Data:", DATA, "| train:", len(labels), "| test:", len(sample), "| device:", device)


In [ ]:
class Clips(Dataset):
    def __init__(self, root, split, ids, transform):
        self.root, self.split, self.ids, self.transform = root, split, list(ids), transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        folder = self.root / self.split / self.ids[i]
        frames, gray = [], []
        for j in range(30):
            with Image.open(folder / f"frame_{j:03d}.jpg") as raw:
                image = ImageOps.exif_transpose(raw).convert("RGB")
                frames.append(self.transform(image))
                gray.append(np.asarray(image.convert("L").resize((64, 36)), dtype=np.float32))
        motion = np.abs(np.diff(np.stack(gray), axis=0)).mean(axis=(1, 2)) / 255.0
        return torch.stack(frames), torch.from_numpy(motion)

weights = ResNet50_Weights.IMAGENET1K_V2
try:
    backbone = resnet50(weights=weights)
except Exception as e:
    raise RuntimeError("ResNet50 ImageNet weights could not load. Enable Kaggle Internet and rerun.") from e
backbone.fc = nn.Identity()
backbone = backbone.to(device).eval()
for p in backbone.parameters():
    p.requires_grad_(False)

@torch.inference_mode()
def extract(split, ids):
    ds = Clips(DATA, split, ids, weights.transforms())
    loader = DataLoader(ds, batch_size=4, num_workers=2 if device.type == "cuda" else 0,
                        pin_memory=device.type == "cuda")
    features, motions = [], []
    start = time.time()
    for n, (images, motion) in enumerate(loader, 1):
        b, t = images.shape[:2]
        amp = torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else nullcontext()
        with amp:
            feat = backbone(images.reshape(b * t, *images.shape[2:]).to(device))
        features.append(feat.reshape(b, t, -1).cpu().half())
        motions.append(motion)
        if n % 100 == 0:
            print(split, n * loader.batch_size, "/", len(ds), "clips,", round(time.time() - start), "s", flush=True)
    return torch.cat(features), torch.cat(motions).numpy()

train_f, train_motion = extract("train", labels.clip_id)
test_f, test_motion = extract("test", sample.clip_id)
assert train_f.shape == (len(labels), 30, 2048)
assert test_f.shape == (len(sample), 30, 2048)
assert np.isfinite(train_motion).all() and np.isfinite(test_motion).all()
del backbone
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
print("Features ready:", tuple(train_f.shape), tuple(test_f.shape))


In [ ]:
class CrashHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm = nn.LayerNorm(2048)
        self.project = nn.Sequential(nn.Linear(2048, 256), nn.GELU())
        self.classifier = nn.Sequential(nn.Dropout(0.35), nn.Linear(1024, 128),
                                        nn.GELU(), nn.Dropout(0.35), nn.Linear(128, 1))

    def forward(self, x):
        h = self.project(self.norm(x))
        motion = (h[:, 1:] - h[:, :-1]).abs().amax(dim=1)
        summary = torch.cat((h.mean(dim=1), h.amax(dim=1),
                             h[:, -1] - h[:, 0], motion), dim=1)
        return self.classifier(summary).squeeze(1)

# Quick executable checks for tensor shape and group isolation.
assert CrashHead()(torch.zeros(2, 30, 2048)).shape == (2,)
folds = list(GroupKFold(5).split(labels, labels.label, labels.group_id))
assert all(set(labels.group_id.iloc[tr]).isdisjoint(labels.group_id.iloc[va])
           for tr, va in folds)
print("Head and group split checks passed.")


In [ ]:
torch.manual_seed(42)
np.random.seed(42)
x_train = train_f.float().to(device)
x_test = test_f.float().to(device)
y = labels.label.to_numpy(dtype=np.float32)
deep_oof = np.full(len(labels), np.nan)
motion_oof = np.full(len(labels), np.nan)
deep_test = np.zeros(len(sample))
motion_test = np.zeros(len(sample))

@torch.inference_mode()
def predict_head(model, x, batch_size=128):
    model.eval()
    return np.concatenate([torch.sigmoid(model(z)).cpu().numpy()
                           for z in x.split(batch_size)])

for fold, (tr, va) in enumerate(folds, 1):
    model = CrashHead().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=7e-4, weight_decay=0.05)
    best_auc, best_state, wait = -1.0, None, 0
    for epoch in range(50):
        model.train()
        shuffled = np.random.permutation(tr)
        for batch in np.array_split(shuffled, range(128, len(shuffled), 128)):
            target = torch.as_tensor(y[batch], device=device)
            logits = model(x_train[batch])
            loss = nn.functional.binary_cross_entropy_with_logits(logits, target)
            opt.zero_grad()
            loss.backward()
            opt.step()
        pred = predict_head(model, x_train[va])
        auc = roc_auc_score(y[va], pred)
        if auc > best_auc + 1e-4:
            best_auc, wait = auc, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= 8:
                break
    model.load_state_dict(best_state)
    deep_oof[va] = predict_head(model, x_train[va])
    deep_test += predict_head(model, x_test) / len(folds)

    motion_model = HistGradientBoostingClassifier(max_iter=150, max_leaf_nodes=15,
        learning_rate=0.05, l2_regularization=3, random_state=fold)
    motion_model.fit(train_motion[tr], y[tr].astype(int))
    motion_oof[va] = motion_model.predict_proba(train_motion[va])[:, 1]
    motion_test += motion_model.predict_proba(test_motion)[:, 1] / len(folds)
    print(f"Fold {fold}: image AUC {best_auc:.5f}; motion AUC "
          f"{roc_auc_score(y[va], motion_oof[va]):.5f}; epoch {epoch + 1}", flush=True)
    del model, motion_model
    if device.type == "cuda":
        torch.cuda.empty_cache()

assert np.isfinite(deep_oof).all() and np.isfinite(motion_oof).all()
print("Grouped OOF image AUC:", round(roc_auc_score(y, deep_oof), 5))
print("Grouped OOF motion AUC:", round(roc_auc_score(y, motion_oof), 5))


In [ ]:
weights_to_try = (0.0, 0.25, 0.5, 0.75, 1.0)
scores = {w: roc_auc_score(y, w * deep_oof + (1 - w) * motion_oof)
          for w in weights_to_try}
weight = max(scores, key=scores.get)
print("Grouped OOF blend AUCs:", {w: round(s, 5) for w, s in scores.items()})
print("Selected image weight:", weight, "| grouped OOF AUC:", round(scores[weight], 5))

submission = sample[["clip_id"]].copy()
submission["label"] = np.clip(weight * deep_test + (1 - weight) * motion_test, 1e-6, 1 - 1e-6)
assert submission.columns.tolist() == ["clip_id", "label"]
assert submission.clip_id.is_unique and submission.clip_id.tolist() == sample.clip_id.tolist()
assert submission.label.notna().all() and submission.label.between(0, 1).all()
output = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
submission.to_csv(output, index=False)
print("Wrote", output, "with", len(submission), "rows")
print(submission.label.describe().round(4).to_string())
